# Solar Flux Prediction

In [1]:
import torch

device_try = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device: {device_try}")

Selected device: cuda


In [2]:
import os

print("=== TEST TMUX ===")
tmux_var = os.environ.get("TMUX")

if tmux_var:
    print("STATUS: Jupyter server is running in a TMUX session!")
    print(f"Tmux socket path: {tmux_var}")
else:
    print("STATUS: The Jupyter server is NOT inside Tmux (it runs in the global terminal).")

=== TEST TMUX ===
STATUS: Jupyter server is running in a TMUX session!
Tmux socket path: /tmp//tmux-1022/default,842946,5


In [3]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [4]:
print("--- Caricamento e Aggregazione Giornaliera ---")
df = pd.read_csv("../Data/MARSIS_historical_dataset.csv", sep=";")
df.columns = df.columns.str.strip()

# 1. Conversione tempo
df['datetime'] = pd.to_datetime(df['FM_data_ephemeris_time'], unit='s', origin=pd.Timestamp('2000-01-01 12:00:00'))

# 2. Aggregazione Giornaliera
df_daily = df.groupby(df['datetime'].dt.to_period('D')).agg({
    'FM_data_F10_7_index': 'mean'
}).reset_index()

df_daily['datetime'] = df_daily['datetime'].dt.to_timestamp()

# 3. FIX: Riempimento giorni mancanti PRIMA di estrarre le feature temporali
df_daily = df_daily.set_index('datetime').asfreq('D', method='ffill').reset_index()

# 4. Estrazione feature temporali (ora su un calendario perfetto e senza buchi)
df_daily['day_of_year'] = df_daily['datetime'].dt.dayofyear
df_daily['month_of_year'] = df_daily['datetime'].dt.month 

print(f"Dataset giornaliero creato. Totale giorni disponibili: {len(df_daily)}")

--- Caricamento e Aggregazione Giornaliera ---
Dataset giornaliero creato. Totale giorni disponibili: 5532


In [5]:
import numpy as np
from time_series_utils import *

LOOKBACK_WINDOW = 30 

# Creiamo le matrici usando il dataset giornaliero e il giorno dell'anno
X_flux, X_time, y = create_dataset_windows(
    data=df_daily, 
    target_col='FM_data_F10_7_index', 
    time_col='day_of_year',  
    lookback=LOOKBACK_WINDOW
)

# Split Cronologico (80% Train, 20% Test)
split_idx = int(len(y) * 0.8)

X_flux_train, X_flux_test = X_flux[:split_idx], X_flux[split_idx:]
X_time_train, X_time_test = X_time[:split_idx], X_time[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

## Linear regression for solar flux prediction 

In [6]:
print("\n--- Training: Regressione Lineare ---")

# Uniamo le feature (Flusso_t-30...Flusso_t + Giorno_t-30...Giorno_t)
X_linear_train = np.hstack((X_flux_train, X_time_train))
X_linear_test = np.hstack((X_flux_test, X_time_test))

# Inizializzazione e fit
lr_model = LinearRegression()
lr_model.fit(X_linear_train, y_train)

# Predizione e Valutazione
y_pred_lr = lr_model.predict(X_linear_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
print(f"[Linear Regression] Test MAE: {mae_lr:.3f}, Test RMSE: {rmse_lr:.3f}")


--- Training: Regressione Lineare ---
[Linear Regression] Test MAE: 0.401, Test RMSE: 1.352


## LSTM for solar flux prediction